# Prepare a request of imagery at sites from a vendor using the CSDA Evaluation Sites GeoJSON

Paul Montesano, PhD  
June 2026

In [7]:
import pandas as pd
import geopandas as gpd
from datetime import datetime

### Read the CSDA Sites GeoJSON stored on GitHub

+ This GeoJSON is built directly off the CSDA Evaluation Sites Database.  
+ The notebook to process this GeoJSON is here: https://github.com/pahbs/csda_summaries/blob/master/notebooks/csda_eval_sites_process.ipynb

In [51]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
sites_url = f'{RAW_BASE}/sites/csda_sites_aoi.geojson'
sites = gpd.read_file(sites_url)

### Indicate a name for the vendor

In [52]:
VENDOR_NAME = 'Tanager' # Change this

In [53]:
# Get today's date
DATE = datetime.now().strftime('%Y%m%d')
DATE

'20260609'

In [54]:
#sites.info()

### Check some useful site attributes

In [55]:
print(list(sites['Evaluation Category'].unique()))

['Geometric', 'Radiometric', 'Radiometric & Geometric', 'InSAR']


### Each site's 'Site Name' is the key identifier for indicating the location of a CSDA request of data from a vendor

In [56]:
print(list(sites['Site Name'].unique()))

['Albuquerque', 'Amazon', 'Baotou', 'Atacama Desert', 'Belo Horizonte', 'Boston', 'Cairo', 'Cape Town', 'Casablanca', 'Caspian Sea', 'Catania', 'Crater Lake', 'Cuprite', 'Antarctica GPS', 'Doldrums', 'Atlantic Doldrums', 'Dublin', 'Gobabeb', 'Petermann Glacier', 'NISAR CR Array', 'Hohhot', 'Tianjin Docks', 'San Mateo Bridge', 'King Fahd Causeway', 'La Crau', 'Lake Pontchartrain Causeway', 'London', 'Melbourne', 'Navarre Causeway', 'DLR CR Array', 'Arabian Peninsula', 'Old Bahia Bridge', 'Phoenix', 'PICS Algeria-3', 'PICS Libya-1', 'PICS Libya-4', 'Piedmont', 'Railroad Valley', 'Buenos Aires', 'Rio Gallegos', 'RCRA', 'Salon-de-Provence', 'Sapporo', 'Suramadu Bridge', 'Shadnagar', 'Singapore', 'Sioux Falls', 'FMI CR Array', 'Valencia', 'Golmud', 'OPERA CR Array', 'WLEF', 'Etang de Berre', 'La Crau TIR', 'Lageren', 'Lake Constance', 'Lake Kasumigaura', 'Lake Tahoe', 'Myall Vale A', 'Oklahoma Agriculture Station', 'Pinnacles', 'Santarem', 'Salton Sea', 'Russell Ranch', 'Venice', 'AZ Coconi

#### Function to update site attributes

In [61]:
def update_sites_attributes(sites_gdf, site_configs):
    """
    Update sites GeoDataFrame with attributes based on configuration.
    
    Parameters:
    -----------
    sites_gdf : GeoDataFrame
        Sites geodataframe to update
    site_configs : list of dict
        List of configurations, each with 'sites' and 'order_parameters' keys
        
    Returns:
    --------
    GeoDataFrame : Updated sites (copy)
    list : All site names from configs
    """
    sites_updated = sites_gdf.copy()
    all_sites = []
    
    for config in site_configs:
        site_list = config['sites']
        attributes = config['order_parameters']
        
        # Update attributes for these sites
        mask = sites_updated['Site Name'].isin(site_list)
        for key, value in attributes.items():
            sites_updated.loc[mask, key] = value
        
        all_sites.extend(site_list)
    
    return sites_updated, all_sites

def print_site_update_report(sites_original, sites_updated, site_configs):
    """
    Print a report showing what attributes were updated for which sites.
    
    Parameters:
    -----------
    sites_original : GeoDataFrame
        Original sites before updates
    sites_updated : GeoDataFrame
        Sites after updates
    site_configs : list of dict
        Configuration used for updates
    """
    print("=" * 70)
    print("SITE ATTRIBUTE UPDATE REPORT")
    print("=" * 70)
    
    # Get all unique attributes being updated
    all_attributes = set()
    for config in site_configs:
        all_attributes.update(config['order_parameters'].keys())
    
    total_sites = 0
    
    for i, config in enumerate(site_configs, 1):
        sites_list = config['sites']
        attributes = config['order_parameters']
        
        print(f"\nGroup {i}: {len(sites_list)} site(s)")
        print("-" * 70)
        
        for site in sites_list:
            total_sites += 1
            print(f"\n  Site: {site}")
            
            # Get before/after values
            orig_row = sites_original[sites_original['Site Name'] == site]
            updated_row = sites_updated[sites_updated['Site Name'] == site]
            
            if len(orig_row) == 0:
                print(f"    ⚠️  WARNING: Site not found in original dataframe")
                continue
            
            for attr, new_value in attributes.items():
                old_value = orig_row[attr].iloc[0] if attr in orig_row.columns else 'N/A'
                actual_value = updated_row[attr].iloc[0] if len(updated_row) > 0 else 'ERROR'
                
                # Check if update was successful
                if str(actual_value) == str(new_value):
                    status = "✓"
                else:
                    status = "✗"
                
                print(f"    {status} {attr:20s}: {old_value} → {new_value}")
    
    print("\n" + "=" * 70)
    print(f"Total sites updated: {total_sites}")
    print("=" * 70)


### Update config of request parameters for sites chosen for this vendor request

Here is where we config & specify our 'timeseries' sites and any other types of sites we need to config & specify for this request

In [62]:
# This SITE_CONFIGS dictionary gives us our final list of sites and their order parameters for this request
SITE_CONFIGS = [
    {
        'sites': ['Albuquerque', 'Casablanca'],
        'order_parameters': {
            'ideal_num_acqs': 10,
            'request_type': 'timeseries',
            'assessment_domain': 'geometric'
        }
    },
    # Just examples of other configs
    {
        'sites': ['Baotou'],
        'order_parameters': {
            'ideal_num_acqs': 5,
            'request_type': 'other',
            'assessment_domain': 'geometric'
        }
    },
    {
        'sites': ['WLEF', 'PICS Libya-4'],
        'order_parameters': {
            'ideal_num_acqs': 3,
            'request_type': 'other',
            'assessment_domain': 'radiometric'
        }
    }
]

sites_updated, SITES_FOR_REQUEST = update_sites_attributes(sites, SITE_CONFIGS)

print_site_update_report(sites, sites_updated, SITE_CONFIGS)

SITE ATTRIBUTE UPDATE REPORT

Group 1: 2 site(s)
----------------------------------------------------------------------

  Site: Albuquerque
    ✗ ideal_num_acqs      : 10.0 → 10
    ✓ request_type        : N/A → timeseries
    ✓ assessment_domain   : N/A → geometric

  Site: Casablanca
    ✗ ideal_num_acqs      : 5.0 → 10
    ✓ request_type        : N/A → timeseries
    ✓ assessment_domain   : N/A → geometric

Group 2: 1 site(s)
----------------------------------------------------------------------

  Site: Baotou
    ✗ ideal_num_acqs      : nan → 5
    ✓ request_type        : N/A → other
    ✓ assessment_domain   : N/A → geometric

Group 3: 2 site(s)
----------------------------------------------------------------------

  Site: WLEF
    ✗ ideal_num_acqs      : nan → 3
    ✓ request_type        : N/A → other
    ✓ assessment_domain   : N/A → radiometric

  Site: PICS Libya-4
    ✗ ideal_num_acqs      : nan → 3
    ✓ request_type        : N/A → other
    ✓ assessment_domain   : N/A → 

### Create and write subset GeoJSON for this request

In [65]:
sites_subset = sites_updated[sites_updated['Site Name'].isin(SITES_FOR_REQUEST)]
sites_subset

,Site Name abbrev,Site Name,Location Name,Country,Program Use,Longitude,Latitude,Remote Sensing Domain,Priority Level,Evaluation Category,...,cr:id,cr:lat,cr:lon,cr:height_above_ellipsoid_m,cr:orientation_deg,cr:elevation_angle_d,cr:size_m,geometry,request_type,assessment_domain
0,Albuquerque,Albuquerque,New Mexico,USA,CSDA,-106.613826,35.068706,Optical Multi/Hyper,high,Geometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-106.59712 35.05540, -106.59764 35.0...",timeseries,geometric
2,Baotou,Baotou,China cal/val,China,CSDA,109.629437,40.851787,Optical Multi/Hyper,high,Radiometric & Geometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((109.62968 40.85161, 109.62967 40.851...",other,geometric
10,Casablanca,Casablanca,Casablanca,Morocco,CSDA,-7.622420,33.580370,Optical Multi/Hyper,high,Geometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-7.60648 33.56666, -7.60604 33.59371...",timeseries,geometric
37,PICS Libya-4,PICS Libya-4,PICS Libya-4,Libya,CSDA,23.390000,28.550000,Optical Multi/Hyper,high,Radiometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((23.39665 28.55833, 23.39110 28.53333...",other,radiometric
53,WLEF,WLEF,WLEF tower,USA,CSDA,-90.273200,45.944900,Optical Multi/Hyper,high,Radiometric,...,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-90.23454 45.94397, -90.23486 45.941...",other,radiometric


In [ ]:
OUTPUT_DIR = '/my/output/dir' # Specify your output dir here

In [ ]:
sites_subset.to_file(f'{OUTPUT_DIR/csda_sites_aoi_{VENDOR_NAME}_{DATE}.geojson}')